In [1]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=1)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [ ]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=2)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [ ]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    position_x = stage_settings["position_x"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    try:
        bcp303_z = BPC303(channel_id=1)
        bcp303_device = bcp303_z.get_device()
        # moving controller x
        bcp303_x = BPC303(channel_id=2, device=bcp303_device)
        # Sourcemeter
        sm2401 = Sourcemeter2401(speed_nplc=0.1)
        # set the initial position of the stage z
        position_z = bcp303_z.move_to_origin(start_position=stage_settings["position_z"]) - step_size_z
        time.sleep(1)
        bcp303_x.move_to_origin()
        time.sleep(1)
        allData = [{"position": [], "voltage": []}]
        for _ in range(step_number):
            position_z = bcp303_z.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position_z,
            )
            allData[0]["position"].append(position_z)
            time.sleep(0.5)
            bcp303_x.move_to_position(start_position=0, step_size=0.5, final_position=position_x)
            time.sleep(0.5)
            step_start = time.perf_counter()
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            bcp303_x.move_to_origin()
            time.sleep(0.5)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_x"] = position_x
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303_x.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "position_x": 3,
        "step_number": 15,
        "position_z": 0,
        "step_size_z": 1,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        chip_name="V1_R_W_2_Right",
        sample_name="w20_test_sweep",  
    )

APT High Voltage Amplifier initialized
APT High Voltage Amplifier initialized
IDN: KEITHLEY INSTRUMENTS INC.,MODEL 2401,4614233,B02 Jan 20 2021 10:19:49/B01  /W/N
process completed: 6.7%
process completed: 13.3%
process completed: 20.0%
process completed: 26.7%
process completed: 33.3%
process completed: 40.0%
process completed: 46.7%
process completed: 53.3%
process completed: 60.0%
process completed: 66.7%
process completed: 73.3%
process completed: 80.0%
process completed: 86.7%
process completed: 93.3%
process completed: 100.0%
Config saved to: ./result/V1_R_W_2_Right/202606091638_w10_3_sweep\202606091638_V1_R_W_2_Right_w10_3_sweep_config.json
Exception occurred: list indices must be integers or slices, not NoneType
Stage Moving Done
Stage disconnected


({'position': [0.0,
   0.996734519486068,
   1.99407940916166,
   2.9932554094058,
   3.97473067415387,
   4.95132297738578,
   5.94500564592425,
   6.9380779442732,
   7.93664357432783,
   8.93704031495102,
   9.93621631519516,
   10.9323404644917,
   11.9290749839778,
   12.9245887630848,
   13.924985503708],
  'voltage': [[0.03271237,
    0.03260491,
    0.03216136,
    0.03239842,
    0.03258096,
    0.03237322,
    0.0323309,
    0.03233921,
    0.03260291,
    0.03256267,
    0.03258467,
    0.03289386,
    0.03253653,
    0.03260601,
    0.03270733,
    0.03245568],
   [0.03168498,
    0.03169173,
    0.03146267,
    0.03131333,
    0.03168076,
    0.03142172,
    0.03110245,
    0.0313824,
    0.03136047,
    0.03140466,
    0.0312732,
    0.03139395,
    0.03120792,
    0.03115866,
    0.0316484,
    0.03160321],
   [0.03299353,
    0.0328389,
    0.03277093,
    0.03262402,
    0.0330455,
    0.03292626,
    0.0330205,
    0.0328772,
    0.03279651,
    0.03292903,
    0.0328

<Figure size 640x480 with 0 Axes>